# PatchSorter — Database Initialization

End-to-end setup for a new PatchSorter database:

1. Create schema and distribute tables (Citus reference tables)
2. Seed application-level settings
3. Create a placeholder project (project-level settings seeded automatically)
4. Create per-project distributed tables and install confusion-matrix triggers
5. Add 10 label classes to the project
6. Register one whole-slide image
7. Extract patches from a GeoJSON file and load them into the database

> **Prerequisites:**
> - Docker stack running: `docker-compose -f deployment/docker-compose.yaml up -d`
> - GeoJSON features contain a `uid` field (run `add_uuids_to_geojson.ipynb` first if needed)
> - `IMAGE_FILEPATH` points to a `large_image`-readable whole-slide image

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit this cell before running
# ---------------------------------------------------------------------------

IMAGE_FILEPATH      = "/Users/jackson/Research/data/fan_annotations/13_266069_040_003_L02_PAS.ndpi"       # large_image-readable WSI
GEOJSON_FILEPATH    = "/Users/jackson/Research/data/fan_annotations/nuclei_split.geojson"  # must have 'uid' on each feature

PROJECT_NAME        = "Placeholder Project"
PROJECT_DESCRIPTION = "Initial placeholder project for development"

DEEPZOOM_TILESIZE = 256     # tile size for DeepZoom serving

# Patch extraction parameters
DOWNSAMPLE_FACTOR = 2.0  # 1.0 = base mag, 2.0 = half base mag, etc.
PATCH_SIZE        = 256  # output patch edge length in pixels at extraction mag

# Ten cell subtype label classes: (name, CSS hex colour)
LABEL_CLASSES = [
    ("Epithelial",   "#E74C3C"),
    ("Lymphocyte",   "#3498DB"),
    ("Plasma Cell",  "#9B59B6"),
    ("Macrophage",   "#E67E22"),
    ("Neutrophil",   "#F1C40F"),
    ("Eosinophil",   "#1ABC9C"),
    ("Fibroblast",   "#2ECC71"),
    ("Endothelial",  "#E91E63"),
    ("Mast Cell",    "#884400"),
    ("Tumor Cell",   "#C0392B"),
]

In [ ]:
from patchsorter.db.head_client import (
    get_client,
    ImageStore,
    LabelClassStore,
    ProjectStore,
)
from patchsorter.db.head_client.database_manager import DatabaseManager
from patchsorter.utils.patch_extraction import _makepatch_geojson

In [ ]:
# Connect to the head node and initialise the schema.
# setup_schema() creates all base tables, distributes them as Citus reference
# tables, seeds the reserved "unassigned" label class, and seeds application-
# level settings from settings_defaults.toml.
client = get_client()
db_mgr = DatabaseManager(client)


In [ ]:
db_mgr.drop_all_tables()  # drop existing tables for a clean slate


In [ ]:
db_mgr.setup_schema()
print("Schema ready.")

In [ ]:
# Create the project.  ProjectStore.create() also seeds project-scoped
# settings (world_size, agg_hierarchy_depth) via SettingsStore.seed_project_settings.
with client.get_session() as session:
    project_store = ProjectStore(session)
    proj = project_store.create(PROJECT_NAME, PROJECT_DESCRIPTION)

project_id = proj["project_id"]
print(f"Project created  project_id={project_id}  name={proj['project_name']}")

# Create per-project distributed tables (patch, pred_patch_latest/last,
# confusion_matrix_l8..l12) and install per-shard triggers.
db_mgr.setup_project(project_id)
print(f"Per-project tables and triggers ready for project {project_id}.")

In [ ]:
# Add 10 label classes to the project.
# label_class_id=1 is the reserved "unassigned" class seeded at schema time.
# User-defined classes start at id=2.
with client.get_session() as session:
    lc_store = LabelClassStore(session)
    label_ids: dict[str, int] = {}
    for name, color in LABEL_CLASSES:
        lc = lc_store.create(project_id, name, color)
        label_ids[name] = lc["label_class_id"]
        print(f"  label_class_id={lc['label_class_id']:3d}  {name}  ({color})")

print(f"\n{len(label_ids)} label classes created.")

In [ ]:
# Read image properties from the WSI via large_image.
import large_image

_ts = large_image.open(IMAGE_FILEPATH)
_meta = _ts.getMetadata()

BASE_MAG    = _meta["magnification"]
BASE_WIDTH  = _meta["sizeX"]
BASE_HEIGHT = _meta["sizeY"]

print(f"BASE_MAG={BASE_MAG}x  BASE_WIDTH={BASE_WIDTH}px  BASE_HEIGHT={BASE_HEIGHT}px")

In [ ]:
# Register the whole-slide image.
with client.get_session() as session:
    img = ImageStore(session).create(
        project_id=project_id,
        name="placeholder_image",
        image_path=IMAGE_FILEPATH,
        base_mag=BASE_MAG,
        base_width=BASE_WIDTH,
        base_height=BASE_HEIGHT,
        deepzoom_tilesize=DEEPZOOM_TILESIZE,
    )

image_id = img["image_id"]
print(f"Image registered  image_id={image_id}  path={img['image_path']}")

In [ ]:
# Extract patches from the GeoJSON file and load them into the database.
#
# All patches are assigned to the first user-defined label class by default.
# Change PATCH_LABEL_CLASS to the name of whichever class is appropriate,
# or supply an integer label_class_id directly.
PATCH_LABEL_CLASS = LABEL_CLASSES[0][0]  # "Tumor" — change as needed
patch_label_class_id = label_ids[PATCH_LABEL_CLASS]

with client.get_session() as session:
    n_inserted = _makepatch_geojson(
        image_filepath=IMAGE_FILEPATH,
        geojson_filepath=GEOJSON_FILEPATH,
        project_id=project_id,
        image_id=image_id,
        label_class_id=patch_label_class_id,
        session=session,
        patch_size=PATCH_SIZE,
        downsample_factor=DOWNSAMPLE_FACTOR,
    )

print(f"Patches inserted: {n_inserted}  (label_class='{PATCH_LABEL_CLASS}', id={patch_label_class_id})")

In [ ]:
db_mgr.clear_predictions(1)

In [ ]:
# ---------------------------------------------------------------------------
# Initialize Ray cluster and start the deep learning actor
# ---------------------------------------------------------------------------

import ray
from patchsorter.dl.training import startup_dl_actor

In [ ]:
# Start Ray cluster (connects to existing local cluster if already running)
if not ray.is_initialized():
    ray.init(dashboard_host="0.0.0.0")

print(f"Ray cluster address: {ray.get_runtime_context()}")

In [ ]:
# Create (or reuse) the named dl_actor and start the distributed training loop.
# This reads dl_num_workers and dl_patches_per_batch from the project settings.
project_id = 1
actor = startup_dl_actor(project_id)
print(f"DL actor started for project {project_id}")

In [ ]:
ray.shutdown()

In [ ]:
from patchsorter.db.worker_client import get_client

head_client = get_client()

In [ ]:
with head_client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT p.shardid 
        FROM pg_dist_placement p
        JOIN pg_dist_node n ON p.groupid = n.groupid
        WHERE n.nodename = 'localhost'
        ORDER BY p.shardid;
    """)
    for row in cur.fetchall():
        print(row)

In [ ]:
with head_client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT * 
        FROM citus_shards 
        WHERE nodename = 'localhost';
    """)
    for row in cur.fetchall():
        print(row)

In [ ]:
with head_client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_patch%';
    """)
    shard_ids = [row[0] for row in cur.fetchall()]
    print(shard_ids)

In [ ]:
with head_client.get_connection() as conn:
    cur = conn.cursor()
    cur.execute("""
        SELECT shardid
        FROM citus_shards 
        WHERE nodename = 'localhost'
          AND table_name::text LIKE 'project1_pred_patch_last%';
    """)
    shard_ids = [row[0] for row in cur.fetchall()]
    shard_ids.sort()
    print(shard_ids)